# Original actor-feature FiLM: EE oscillation audit
Offline analysis of each checkpoint's own Exp2 recordings, primarily pushing/pulling. Selection requires recorded `film_v02_enabled=False`; V02 and non-FiLM recordings are excluded. No policy input perturbations, simulation, training, or model construction.
This tests **consistency with amplification**, not causality. Gait, estimator errors, trunk activity, base motion and actuator saturation remain alternative explanations.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd() if (Path.cwd() / 'notebooks').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'notebooks'))
import exp3_film_oscillation as analysis
CONFIG = {
    'recording_roots': [str(ROOT / 'evaluation/exp2')],
    'tasks': ['b1z1_pact'],  # None discovers every compatible original-FiLM task.
    'checkpoints': None,  # Optional list of recorded SHA256 hashes.
    'checkpoint_paths': {},  # Optional {recorded_sha256: local_checkpoint_path}.
    'evaluation_seeds': [0, 1, 2],
    'profiles': ['pushing', 'pulling'],
    'loading': ['loaded', 'unloaded'],
    'trial_ids': None,  # e.g. [0, 1] for a small audit.
    'sustained_window_seconds': 4.0,
    'event_window_seconds': [-0.5, 2.0],
    'transient_margin_seconds': 0.25,
    'minimum_window_seconds': 2.0,
    'reference_stationarity_tolerance': 0.02,  # Max change in each native sphere coordinate.
    'nperseg': 64, 'overlap_fraction': 0.5, 'min_spectral_segments': 4,
    'frequency_band_hz': [0.5, 12.0],  # None uses all positive Welch frequencies.
    'coherence_threshold': 0.6, 'max_lag_seconds': 0.5,
    'feature_epsilon': 1e-8,
    'head_parity_atol': 3e-5, 'head_parity_rtol': 3e-4,  # Strict CPU/GPU raw-action check; failures are reported, not hidden.
    'minimum_comparison_windows': 6, 'error_match_bin_m': 0.02,
    'pca': True, 'pca_samples': 1000, 'sampling_seed': 0,
    'pca_color': 'signed_error',  # Or load_stage / oscillation_phase (qualitative Hilbert phase).
    'tsne': False, 'tsne_perplexity': 30,  # Optional sklearn; qualitative only.
    'max_plot_windows': 12, 'max_windows': None,  # max_windows limits smoke runs only.
    'output_dir': str(ROOT / 'analysis/exp3_film_oscillation'),
}
CONFIG.update(json.loads(os.environ.get('FILM_OSCILLATION_CONFIG', '{}')))


## Timing, coordinates, and spectral interpretation
Cartesian error is measured minus reference in the yaw-aligned task frame, in metres. Conditioning is command minus estimate: normalized body `[vx, vy, yaw_rate]` and spherical `[radius, elevation, azimuth]`. Measured conditioning is reconstructed at actor time **t**, using the recorded spherical/world reference pair to recover the workspace origin. Successor errors/motion remain at **t+1**.
Only contiguous valid pre-failure segments are analyzed. Short/failed windows and unavailable diagnostics are reported, not padded. Welch uses signed components, constant detrending, Hann windows and the configured overlap; component spectra are averaged, never spectra of vector norms. Coherence requires enough averages; event/moving-reference windows do not receive stationary coherence claims. Lag peaks are exploratory, not significance tests.
`gamma` is residual scaling: `post = (1+gamma)*pre + beta`. Temporal change is exactly `(1+gamma_previous)*delta_pre + delta_gamma*pre_current + delta_beta`. The last two terms form the parameter-change contribution. Their spectral powers are **not additive**, because cross terms remain.

In [ ]:
result = analysis.run_analysis(CONFIG)
print(f"Analyzed {len(result['windows'])} windows from {len(result['provenance'])} recordings")
summary = pd.DataFrame(result['summary'])
if len(summary):
    metrics = ['ee_cartesian_error/oscillatory_rms', 'ee_cartesian_error/peak_to_peak', 'relative_modulation_rms', 'position_film/oscillatory_rms', 'torque_film/oscillatory_rms', 'parameter_change/band_power']
    display(summary[summary.metric.isin(metrics)])
display(pd.DataFrame(result['exclusions']))


## Recorded feature/action associations
Figures show individual contiguous windows with trial IDs, not zero-filled episode averages. Raw numeric traces/PSD and spectral relationships are exported for every retained window; only the first configured number are plotted. Tables average windows within trials, then trials within evaluation seeds, then evaluation seeds within each checkpoint. Sample SD is undefined for one evaluation seed. Checkpoints stay separate; evaluation seeds are not independent training runs.
Linear-head attribution requires a matching checkpoint hash and reconstruction of recorded deterministic actions. CPU/GPU arithmetic may differ; the configured tolerances and maximum mismatch are reported, and strict failures retain feature-only results. Do not raise tolerances without investigating. Physical command effects are in rad/Nm before command clipping. Nominal torque effects, when mechanics are available, apply clipping and measured-state PD algebra; they ignore action delay/substeps and are **not executed torque or simulated counterfactuals**. Recorded executed interval means are labeled separately.
High/low comparisons are stratified by checkpoint/profile/load/motion stage, with explicit error-magnitude bins. PCA uses one centered pooled pre/post basis per checkpoint with paired points; explained variance is exported. t-SNE is disabled by default and cannot establish feedback amplification.

In [ ]:
OUT = Path(CONFIG['output_dir'])
for path in sorted(OUT.glob('window-*-*.png'))[:4]:
    display(Image(filename=str(path)))
display(pd.DataFrame(result['pca']).T)
for path in sorted(OUT.glob('pca-*.png')):
    display(Image(filename=str(path)))
print('Exports:', OUT)


## Dependencies and outputs
CPU: NumPy, SciPy, pandas, matplotlib, Jupyter. Optional PyTorch only deserializes frozen head weights; optional sklearn supplies t-SNE. Isaac Lab and BARD are never initialized.
`summary.json` stores configuration, provenance, units, exclusions, window completeness and checkpoint/seed summaries. CSVs contain window/trial/seed summaries, high/low comparisons and frequency/lag relationships. `window-*.npz` contains timestamped compact signals/PSDs, with PDF/PNG figures. `COMPLETE` is written last. Use a fresh output directory for distinct selections; no raw recording is modified.